# LeetCode 51 — N-Queens (Hard): nhập môn Backtracking bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def solveNQueens(self, n: int) -> List[List[str]]:

```

Đây là bài "kinh điển" để học **backtracking** (quay lui): thử một lựa chọn, đi tiếp, nếu bế tắc thì **gỡ lựa chọn** và thử cái khác.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì và cách biểu diễn bàn cờ gọn bằng một list `queens`.
2. Vì sao brute force thử mọi hoán vị phải duyệt tới `n!` trường hợp.
3. Đệ quy (recursion), call stack, cây quyết định và khuôn mẫu **chọn → đi tiếp → bỏ chọn**.
4. Backtracking theo hàng, kiểm tra an toàn bằng cách quét các hậu đã đặt.
5. Mẹo đặc trưng đường chéo bằng `r - c` và `r + c`, kiểm tra an toàn `O(1)` bằng 3 `set` (cách chính).
6. (Nâng cao) Phiên bản bitmask cực gọn.
7. Lỗi hay gặp khi viết backtracking, mẹo nhận dạng, biến thể (#52 N-Queens II...) và bài tập.

## 1. Đề bài

Bài toán **n-queens** là bài toán đặt `n` quân hậu lên bàn cờ `n x n` sao cho **không có hai quân hậu nào ăn nhau**.

Cho một số nguyên `n`, hãy trả về **tất cả** các lời giải phân biệt của bài toán n-queens. Có thể trả kết quả theo **thứ tự bất kỳ**.

Mỗi lời giải là một cấu hình bàn cờ, trong đó `'Q'` là quân hậu và `'.'` là ô trống.

**Example 1**

```text
Input: n = 4
Output: [[".Q..","...Q","Q...","..Q."],["..Q.","Q...","...Q",".Q.."]]
Explanation: There exist two distinct solutions to the 4-queens puzzle as shown above
```

**Example 2**

```text
Input: n = 1
Output: [["Q"]]
```

**Constraints**

- `1 <= n <= 9`

---

### Quân hậu ăn theo những hướng nào?

Quân hậu đi (và ăn) bao xa cũng được theo **4 loại đường thẳng**:

| Loại đường | Ví dụ ô cùng đường với `(1, 1)` trên bàn 4x4 |
|---|---|
| cùng **hàng** | `(1,0) (1,2) (1,3)` |
| cùng **cột** | `(0,1) (2,1) (3,1)` |
| cùng **đường chéo chính** `\` | `(0,0) (2,2) (3,3)` |
| cùng **đường chéo phụ** `/` | `(0,2) (2,0)` |

Quy ước trong notebook: ô `(r, c)` nghĩa là hàng `r` (row, đếm từ trên xuống, bắt đầu 0) và cột `c` (column, đếm từ trái sang, bắt đầu 0).

In [1]:
from typing import List

example_1 = [[".Q..", "...Q", "Q...", "..Q."], ["..Q.", "Q...", "...Q", ".Q.."]]

for k, board in enumerate(example_1, 1):
    print(f"Lời giải {k}:")
    for row in board:
        print("   ", " ".join(row))   # thêm dấu cách cho dễ nhìn
    print()

Lời giải 1:
    . Q . .
    . . . Q
    Q . . .
    . . Q .

Lời giải 2:
    . . Q .
    Q . . .
    . . . Q
    . Q . .



### Quan sát quan trọng: mỗi hàng có **đúng một** quân hậu

- Hai hậu cùng hàng thì ăn nhau → mỗi hàng có **tối đa** 1 hậu.
- Có `n` hậu và chỉ có `n` hàng → mỗi hàng phải có **đúng** 1 hậu.

Tương tự, mỗi cột cũng có đúng 1 hậu.

Vì vậy thay vì lưu cả bàn cờ, ta chỉ cần một list:

```text
queens[r] = cột của quân hậu ở hàng r
```

Lời giải 1 của ví dụ (`.Q..`, `...Q`, `Q...`, `..Q.`) chính là `queens = [1, 3, 0, 2]`.

Đến lúc trả kết quả mới đổi `queens` thành list chuỗi. Python cho phép **nhân chuỗi** với số nguyên: `'.' * 3` là `'...'`.

In [2]:
print(repr('.' * 3))
print(repr('.' * 1 + 'Q' + '.' * 2))   # hậu ở cột 1 trên bàn 4 cột

def build_board(queens: List[int], n: int) -> List[str]:
    # hàng r: c dấu chấm, rồi 'Q', rồi (n - c - 1) dấu chấm
    return ['.' * c + 'Q' + '.' * (n - c - 1) for c in queens]

print(build_board([1, 3, 0, 2], 4))
print(build_board([1, 3, 0, 2], 4) == example_1[0])

'...'
'.Q..'
['.Q..', '...Q', 'Q...', '..Q.']
True


## 2. Nền tảng 1 — Hoán vị (permutation) và `itertools.permutations`

Vì mỗi **cột** cũng có đúng 1 hậu, list `queens` phải chứa mỗi số `0, 1, ..., n-1` **đúng một lần**. Một cách sắp xếp như vậy gọi là **hoán vị** (permutation).

Ví dụ với `n = 3`, các hoán vị của `[0, 1, 2]` là: `(0,1,2) (0,2,1) (1,0,2) (1,2,0) (2,0,1) (2,1,0)` — có `3! = 6` cái.

Thư viện chuẩn `itertools` có sẵn hàm `permutations(iterable)` sinh lần lượt mọi hoán vị (mỗi hoán vị là một `tuple`).

Số hoán vị là `n! = 1 * 2 * ... * n`, tăng **cực nhanh**.

In [3]:
from itertools import permutations
from math import factorial

print(list(permutations(range(3))))
for n in [4, 6, 8, 9, 12]:
    print(f"n = {n:>2}: n! = {factorial(n):,}")

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]
n =  4: n! = 24
n =  6: n! = 720
n =  8: n! = 40,320
n =  9: n! = 362,880
n = 12: n! = 479,001,600


## 3. Cách 1 — Brute force: thử mọi hoán vị, lọc đường chéo

Nếu `queens` là một hoán vị thì **hàng** và **cột** đã tự động không trùng. Chỉ còn phải kiểm tra **đường chéo**.

Hai ô `(r1, c1)` và `(r2, c2)` nằm cùng một đường chéo khi đi xuống bao nhiêu hàng thì cũng lệch sang bấy nhiêu cột:

```text
|r1 - r2| == |c1 - c2|
```

Ví dụ:

| Ô 1 | Ô 2 | `|dr|` | `|dc|` | Cùng chéo? |
|---|---|---:|---:|---|
| `(0, 0)` | `(2, 2)` | 2 | 2 | có (chéo `\`) |
| `(1, 3)` | `(3, 1)` | 2 | 2 | có (chéo `/`) |
| `(0, 1)` | `(2, 0)` | 2 | 1 | không |

Hàm `abs(x)` trả về giá trị tuyệt đối của `x`.

In [4]:
def same_diagonal(r1, c1, r2, c2):
    return abs(r1 - r2) == abs(c1 - c2)

print(same_diagonal(0, 0, 2, 2))
print(same_diagonal(1, 3, 3, 1))
print(same_diagonal(0, 1, 2, 0))

True
True
False


Thuật toán brute force:

1. Duyệt mọi hoán vị `perm` của `range(n)`.
2. Với mỗi cặp hàng `r1 < r2`, nếu hai hậu cùng chéo thì loại.
3. Hoán vị nào sống sót → đổi thành bàn cờ, thêm vào kết quả.

Mẹo Python: `all(...)` trả về `True` nếu **mọi** điều kiện bên trong đều đúng.

In [5]:
def n_queens_bruteforce(n: int) -> List[List[str]]:
    result = []
    for perm in permutations(range(n)):
        ok = all(
            abs(r1 - r2) != abs(perm[r1] - perm[r2])
            for r1 in range(n)
            for r2 in range(r1 + 1, n)
        )
        if ok:
            result.append(build_board(list(perm), n))
    return result

print(n_queens_bruteforce(4))
print(n_queens_bruteforce(1))
print(n_queens_bruteforce(3))   # n = 2, 3 không có lời giải

[['.Q..', '...Q', 'Q...', '..Q.'], ['..Q.', 'Q...', '...Q', '.Q..']]
[['Q']]
[]


In [6]:
print(" n | số hoán vị phải thử | số lời giải")
for n in range(1, 9):
    print(f"{n:>2} | {factorial(n):>19,} | {len(n_queens_bruteforce(n)):>10}")

 n | số hoán vị phải thử | số lời giải
 1 |                   1 |          1
 2 |                   2 |          0
 3 |                   6 |          0
 4 |                  24 |          2
 5 |                 120 |         10
 6 |                 720 |          4
 7 |               5,040 |         40
 8 |              40,320 |         92


### Độ phức tạp và nút thắt của brute force

- Có `n!` hoán vị, mỗi hoán vị kiểm tra `O(n²)` cặp → **Time `O(n! · n²)`**.
- Space: `O(n)` cho một hoán vị đang xét (chưa tính output).

Với `n = 8`: thử 40 320 hoán vị chỉ để lấy 92 lời giải.

**Vướng ở đâu?** Nhìn hoán vị bắt đầu bằng `(0, 1, ...)`: hai hậu đầu `(0,0)` và `(1,1)` đã ăn nhau, nhưng brute force vẫn sinh **toàn bộ** `(n-2)!` hoán vị có cùng phần đầu đó rồi mới loại từng cái.

**Mấu chốt cải tiến:** đặt hậu **từng hàng một** và kiểm tra **ngay** khi đặt. Phần đầu đã sai thì bỏ luôn cả nhánh phía sau. Đó là **backtracking**. Để viết được nó, trước hết cần hiểu **đệ quy**.

## 4. Nền tảng 2 — Đệ quy (recursion) và call stack

**Đệ quy** là khi một hàm **gọi lại chính nó** với bài toán nhỏ hơn. Hàm đệ quy luôn có 2 phần:

1. **Điểm dừng** (base case): trường hợp đủ nhỏ để trả lời ngay.
2. **Bước đệ quy**: giải một phần, rồi gọi chính hàm đó cho phần còn lại.

Mỗi lần gọi hàm, Python đặt một "khung" (frame) chứa biến cục bộ lên **call stack** (ngăn xếp lời gọi). Lời gọi nào vào sau thì kết thúc trước, rồi quay về đúng chỗ của lời gọi cha.

Ví dụ: sinh mọi chuỗi nhị phân độ dài 2. Ta in thụt lề theo độ sâu `depth` để **nhìn thấy** call stack.

In [7]:
def binary_strings(prefix, length, depth=0):
    print("  " * depth + f"vào binary_strings(prefix='{prefix}')")
    if len(prefix) == length:          # điểm dừng
        print("  " * depth + f"-> thu được '{prefix}'")
        return
    for bit in "01":                   # bước đệ quy: thử từng lựa chọn
        binary_strings(prefix + bit, length, depth + 1)

binary_strings("", 2)

vào binary_strings(prefix='')
  vào binary_strings(prefix='0')
    vào binary_strings(prefix='00')
    -> thu được '00'
    vào binary_strings(prefix='01')
    -> thu được '01'
  vào binary_strings(prefix='1')
    vào binary_strings(prefix='10')
    -> thu được '10'
    vào binary_strings(prefix='11')
    -> thu được '11'


Nhìn output như một **cây quyết định** (decision tree):

```text
                ''
           /          \
        '0'            '1'
       /   \          /   \
    '00'  '01'     '10'  '11'
```

- Mỗi **nút** là một trạng thái (phần đã chọn).
- Mỗi **cạnh** là một lựa chọn (thêm `'0'` hay `'1'`).
- Đệ quy đi theo kiểu **DFS** (Depth-First Search — đi sâu trước): xuống tận lá `'00'`, quay lên, rồi mới sang `'01'`...

Mức thụt lề trong output chính là độ cao của call stack lúc đó.

## 5. Nền tảng 3 — Backtracking: chọn → đi tiếp → bỏ chọn

Ở ví dụ trên, mỗi lời gọi tạo chuỗi mới `prefix + bit`. Với bài lớn, người ta thường dùng **một list chung** `path` và sửa trực tiếp nó:

```python
path.append(x)      # 1. CHỌN x
backtrack(...)      # 2. ĐI TIẾP với lựa chọn đó
path.pop()          # 3. BỎ CHỌN, trả path về như cũ để thử lựa chọn khác
```

Bước 3 chính là "quay lui" (backtrack): khôi phục trạng thái để lời gọi cha thử lựa chọn tiếp theo như chưa có gì xảy ra.

Ví dụ mini: sinh mọi hoán vị của `[0, 1, 2]` bằng backtracking (tức là tự viết lại `permutations`).

Lưu ý: khi thu kết quả phải lưu **bản sao** `path[:]`, vì `path` sẽ còn bị sửa tiếp (xem mục Lỗi thường gặp).

In [8]:
def permutations_backtrack(n):
    result, path, used = [], [], set()

    def backtrack():
        if len(path) == n:
            result.append(path[:])     # lưu BẢN SAO
            return
        for x in range(n):
            if x in used:
                continue
            path.append(x); used.add(x)        # chọn
            backtrack()                        # đi tiếp
            path.pop(); used.remove(x)         # bỏ chọn

    backtrack()
    return result

print(permutations_backtrack(3))

[[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]


Đây chính là bộ khung của N-Queens:

| Hoán vị | N-Queens |
|---|---|
| vị trí thứ `k` trong `path` | hàng `r` |
| lựa chọn: số `x` chưa dùng | lựa chọn: cột `c` an toàn |
| điều kiện: `x not in used` | điều kiện: không cùng cột, không cùng 2 đường chéo |
| dừng khi `len(path) == n` | dừng khi đã đặt đủ `n` hàng |

Khác biệt duy nhất: điều kiện chặt hơn. Nhờ kiểm tra **trước khi đi tiếp**, cả một nhánh sai bị cắt ngay — gọi là **cắt tỉa** (pruning).

## 6. Cách 2 — Backtracking theo hàng, kiểm tra an toàn bằng cách quét

Ý tưởng:

```text
backtrack(r):                       # đã đặt xong hàng 0..r-1
    nếu r == n: lưu lời giải, return
    với mỗi cột c = 0..n-1:
        nếu (r, c) an toàn với MỌI hậu đã đặt:
            đặt hậu ở (r, c)        # chọn
            backtrack(r + 1)        # đi tiếp
            nhấc hậu ra             # bỏ chọn
```

Ta không cần kiểm tra **hàng**: mỗi hàng chỉ đặt một hậu theo cách xây dựng.

Hàm `is_safe_scan` quét các hậu đã đặt `(pr, queens[pr])` với `pr < r`: nếu cùng cột hoặc cùng chéo thì không an toàn.

`enumerate(queens)` cho lần lượt cặp `(chỉ số, giá trị)`, tức `(pr, pc)`.

In [9]:
def is_safe_scan(queens: List[int], r: int, c: int) -> bool:
    for pr, pc in enumerate(queens):           # các hậu ở hàng 0..r-1
        if pc == c or abs(pr - r) == abs(pc - c):
            return False
    return True

queens = [0, 2]                      # hậu ở (0,0) và (1,2)
for c in range(4):
    print(f"ô (2,{c}) an toàn? {is_safe_scan(queens, 2, c)}")

ô (2,0) an toàn? False
ô (2,1) an toàn? False
ô (2,2) an toàn? False
ô (2,3) an toàn? False


In [10]:
def n_queens_backtrack_scan(n: int) -> List[List[str]]:
    result = []
    queens = []                      # queens[r] = cột của hậu ở hàng r

    def backtrack(r: int) -> None:
        if r == n:
            result.append(build_board(queens, n))
            return
        for c in range(n):
            if is_safe_scan(queens, r, c):
                queens.append(c)     # chọn
                backtrack(r + 1)     # đi tiếp
                queens.pop()         # bỏ chọn

    backtrack(0)
    return result

print(n_queens_backtrack_scan(4))
print(n_queens_backtrack_scan(1))
print(len(n_queens_backtrack_scan(8)))

[['.Q..', '...Q', 'Q...', '..Q.'], ['..Q.', 'Q...', '...Q', '.Q..']]
[['Q']]
92


### Trace n = 4: vẽ bàn cờ từng bước (đến lời giải đầu tiên)

Mỗi dòng vẽ bàn cờ 4 hàng nằm ngang, cách nhau bởi dấu cách. Thụt lề = độ sâu call stack = số hậu đã đặt.

| Bước | Hành động | Bàn cờ (hàng 0 → 3) | Ghi chú |
|---:|---|---|---|
| 1 | đặt (0,0) | `Q... .... .... ....` | |
| 2 | đặt (1,2) | `Q... ..Q. .... ....` | (1,0) cùng cột, (1,1) cùng chéo |
| 3 | hàng 2 bế tắc | | mọi cột của hàng 2 đều bị (0,0) hoặc (1,2) ăn |
| 4 | nhấc (1,2), đặt (1,3) | `Q... ...Q .... ....` | |
| 5 | đặt (2,1) | `Q... ...Q .Q.. ....` | |
| 6 | hàng 3 bế tắc | | quay lui 2 tầng: hàng 2 và hàng 1 cũng hết cột |
| 7 | nhấc (0,0), đặt (0,1) | `.Q.. .... .... ....` | thử cột tiếp theo của hàng 0 |
| 8 | đặt (1,3), (2,0), (3,2) | `.Q.. ...Q Q... ..Q.` | đủ 4 hậu → lời giải `[1, 3, 0, 2]` |

Hàm `trace_backtrack_scan` in đúng các bước trên (dừng sau `stop_after` lời giải cho output gọn).

In [11]:
def one_line(queens: List[int], n: int) -> str:
    rows = ['.' * c + 'Q' + '.' * (n - c - 1) for c in queens]
    rows += ['.' * n] * (n - len(queens))          # các hàng chưa đặt
    return ' '.join(rows)

def trace_backtrack_scan(n: int, stop_after: int = 1) -> None:
    queens, found = [], []

    def backtrack(r: int) -> None:
        ind = '  ' * r
        if r == n:
            found.append(queens[:])
            print(f"{ind}=> LỜI GIẢI {queens}")
            return
        for c in range(n):
            if is_safe_scan(queens, r, c):
                queens.append(c)
                print(f"{ind}đặt ({r},{c})   {one_line(queens, n)}")
                backtrack(r + 1)
                if len(found) >= stop_after:
                    return                          # chỉ để trace gọn
                queens.pop()
                print(f"{ind}nhấc ({r},{c})")
        print(f"{ind}hàng {r}: hết cột -> quay lui")

    backtrack(0)

trace_backtrack_scan(4)

đặt (0,0)   Q... .... .... ....
  đặt (1,2)   Q... ..Q. .... ....
    hàng 2: hết cột -> quay lui
  nhấc (1,2)
  đặt (1,3)   Q... ...Q .... ....
    đặt (2,1)   Q... ...Q .Q.. ....
      hàng 3: hết cột -> quay lui
    nhấc (2,1)
    hàng 2: hết cột -> quay lui
  nhấc (1,3)
  hàng 1: hết cột -> quay lui
nhấc (0,0)
đặt (0,1)   .Q.. .... .... ....
  đặt (1,3)   .Q.. ...Q .... ....
    đặt (2,0)   .Q.. ...Q Q... ....
      đặt (3,2)   .Q.. ...Q Q... ..Q.
        => LỜI GIẢI [1, 3, 0, 2]


Nhận xét từ trace:

- Nhánh bắt đầu `(0,0)` bị bỏ **sau đúng 3 lần đặt hậu**. Brute force thì phải sinh cả 6 hoán vị bắt đầu bằng `0` rồi mới loại.
- Mỗi dòng `nhấc` ứng với một `queens.pop()`: trạng thái được trả lại y như trước khi đặt.
- Khi một hàng hết cột, hàm `return` → quay về vòng `for` của hàng phía trên, thử cột kế tiếp.

### Độ phức tạp cách 2

- Số nút trên cây quyết định bị cắt tỉa nhiều, nhưng giới hạn trên vẫn thường ghi là `O(n!)` (hàng 0 có ≤ n lựa chọn, hàng 1 ≤ n-1, ...).
- Mỗi lần thử ô `(r, c)` tốn `O(r) = O(n)` để quét → khoảng **`O(n! · n)`** (chưa kể `O(n²)` để dựng mỗi bàn cờ output).
- Space: `O(n)` cho `queens` và độ sâu call stack.

**Vướng ở đâu?** `is_safe_scan` quét lại tất cả hậu đã đặt **cho mỗi ô thử**. Thông tin "cột nào / đường chéo nào đã bị chiếm" bị tính lại liên tục.

**Mấu chốt cải tiến:** ghi nhớ các cột / đường chéo đã bị chiếm trong `set` để hỏi "đã bị chiếm chưa?" trong `O(1)`. Muốn vậy cần **đặt tên** cho mỗi đường chéo bằng một con số.

## 7. Nền tảng 4 — `set` trong Python

`set` (tập hợp) là cấu trúc lưu các giá trị **không trùng nhau**, hỗ trợ:

| Thao tác | Cú pháp | Thời gian trung bình |
|---|---|---|
| tạo rỗng | `s = set()` | |
| thêm | `s.add(x)` | `O(1)` |
| xóa | `s.remove(x)` (lỗi nếu không có) / `s.discard(x)` | `O(1)` |
| kiểm tra có không | `x in s` | `O(1)` |

Chú ý: `{}` là **dict rỗng**, không phải set rỗng.

In [12]:
cols = set()
cols.add(2)
cols.add(0)
cols.add(2)          # thêm trùng: không đổi gì
print(cols)
print(2 in cols, 3 in cols)
cols.remove(2)
print(cols)
print(type({}), type(set()))

{0, 2}
True False
{0}
<class 'dict'> <class 'set'>


## 8. Insight — đặt "tên" cho đường chéo bằng `r - c` và `r + c`

Xem bảng giá trị `r - c` trên bàn 4x4 (hàng `r` theo chiều dọc, cột `c` theo chiều ngang):

```text
r - c      c=0  c=1  c=2  c=3
  r=0       0   -1   -2   -3
  r=1       1    0   -1   -2
  r=2       2    1    0   -1
  r=3       3    2    1    0
```

Các ô **cùng giá trị** xếp thành đường chéo `\`. Lý do: đi xuống-phải một bước thì `r` tăng 1 **và** `c` tăng 1, nên hiệu `r - c` không đổi.

Bảng `r + c`:

```text
r + c      c=0  c=1  c=2  c=3
  r=0       0    1    2    3
  r=1       1    2    3    4
  r=2       2    3    4    5
  r=3       3    4    5    6
```

Các ô cùng giá trị xếp thành đường chéo `/`. Đi xuống-trái một bước: `r` tăng 1, `c` giảm 1, nên tổng `r + c` không đổi.

Ví dụ: `(1,3)` và `(3,1)` có `r + c = 4` → cùng chéo `/`. `(0,1)` và `(2,3)` có `r - c = -1` → cùng chéo `\`.

Kết luận:

- `r - c` là **mã số đường chéo chính** (gọi là `diag`), nhận `2n - 1` giá trị từ `-(n-1)` đến `n-1`.
- `r + c` là **mã số đường chéo phụ** (gọi là `anti`, anti-diagonal), nhận `2n - 1` giá trị từ `0` đến `2n-2`.

In [13]:
n = 4
print("r - c:   ", " ".join(f"c={c}" for c in range(n)))
for r in range(n):
    print(f"   r={r}   ", " ".join(f"{r - c:>3}" for c in range(n)))
print("r + c:   ", " ".join(f"c={c}" for c in range(n)))
for r in range(n):
    print(f"   r={r}   ", " ".join(f"{r + c:>3}" for c in range(n)))

r - c:    c=0 c=1 c=2 c=3
   r=0      0  -1  -2  -3
   r=1      1   0  -1  -2
   r=2      2   1   0  -1
   r=3      3   2   1   0
r + c:    c=0 c=1 c=2 c=3
   r=0      0   1   2   3
   r=1      1   2   3   4
   r=2      2   3   4   5
   r=3      3   4   5   6


Kiểm chứng bằng code: với **mọi** cặp ô khác nhau trên bàn 6x6, công thức cũ `|dr| == |dc|` tương đương với "cùng `r - c` **hoặc** cùng `r + c`".

(Đại số: `|r1 - r2| == |c1 - c2|` nghĩa là `r1 - r2 = c1 - c2` ⇔ `r1 - c1 = r2 - c2`, hoặc `r1 - r2 = -(c1 - c2)` ⇔ `r1 + c1 = r2 + c2`.)

In [14]:
n = 6
cells = [(r, c) for r in range(n) for c in range(n)]
pairs = 0
for (r1, c1) in cells:
    for (r2, c2) in cells:
        if (r1, c1) == (r2, c2):
            continue
        old = abs(r1 - r2) == abs(c1 - c2)
        new = (r1 - c1 == r2 - c2) or (r1 + c1 == r2 + c2)
        assert old == new
        pairs += 1
print("Đã kiểm", pairs, "cặp ô: hai công thức luôn cho cùng kết quả")

Đã kiểm 1260 cặp ô: hai công thức luôn cho cùng kết quả


## 9. Cách 3 (cách chính) — Backtracking với 3 `set`: `cols`, `diag`, `anti`

Ta giữ 3 tập hợp mô tả những đường đã bị chiếm bởi các hậu ở hàng `0..r-1`:

| set | chứa gì | ô `(r, c)` bị ăn nếu |
|---|---|---|
| `cols` | các cột đã có hậu | `c in cols` |
| `diag` | các giá trị `r - c` đã có hậu | `r - c in diag` |
| `anti` | các giá trị `r + c` đã có hậu | `r + c in anti` |

Kiểm tra an toàn giờ chỉ là 3 phép `in` → **`O(1)`**.

Khuôn chọn → đi tiếp → bỏ chọn giờ cập nhật **cả 4 thứ** cùng lúc:

```python
cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)        # chọn
backtrack(r + 1)                                                       # đi tiếp
cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()   # bỏ chọn
```

Bước bỏ chọn phải là **gương** của bước chọn: thêm gì thì gỡ đúng thứ đó.

In [15]:
def n_queens_sets(n: int) -> List[List[str]]:
    result = []
    queens = []
    cols, diag, anti = set(), set(), set()

    def backtrack(r: int) -> None:
        if r == n:                                   # đã đặt đủ n hàng
            result.append(build_board(queens, n))
            return
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue                             # ô bị ăn -> bỏ qua
            # chọn
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            # đi tiếp
            backtrack(r + 1)
            # bỏ chọn
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()

    backtrack(0)
    return result

print(n_queens_sets(4))
print(n_queens_sets(1))
print(n_queens_sets(2))
print([len(n_queens_sets(n)) for n in range(1, 10)])

[['.Q..', '...Q', 'Q...', '..Q.'], ['..Q.', 'Q...', '...Q', '.Q..']]
[['Q']]
[]
[1, 0, 0, 2, 10, 4, 40, 92, 352]


### Trace n = 4 với 3 set (đến lời giải đầu tiên)

Mỗi dòng là một lần ta đứng ở hàng `r`: trạng thái 3 set **trước khi thử**, rồi lý do từng cột bị loại, cho tới khi `ĐẶT` được hoặc hết cột. Sau khi quay lui, hàng đó in thêm một dòng để **thử tiếp** từ cột kế.

| # | r | cols | diag (`r-c`) | anti (`r+c`) | Thử các cột | Kết quả |
|---:|---:|---|---|---|---|---|
| 1 | 0 | {} | {} | {} | c0 | ĐẶT (0,0) |
| 2 | 1 | {0} | {0} | {0} | c0 cột, c1 `1-1=0` ∈ diag, c2 | ĐẶT (1,2) |
| 3 | 2 | {0,2} | {-1,0} | {0,3} | c0 cột, c1 `2+1=3` ∈ anti, c2 cột, c3 `2-3=-1` ∈ diag | hết → quay lui |
| 4 | 1 | {0} | {0} | {0} | (tiếp) c3 | ĐẶT (1,3) |
| 5 | 2 | {0,3} | {-2,0} | {0,4} | c0 cột, c1 | ĐẶT (2,1) |
| 6 | 3 | {0,1,3} | {-2,0,1} | {0,3,4} | c0 cột, c1 cột, c2 `3-2=1` ∈ diag, c3 cột | hết → quay lui |
| 7 | 2 | {0,3} | {-2,0} | {0,4} | (tiếp) c2 `2-2=0` ∈ diag, c3 cột | hết → quay lui |
| 8 | 1 | {0} | {0} | {0} | (tiếp) không còn cột | hết → quay lui |
| 9 | 0 | {} | {} | {} | (tiếp) c1 | ĐẶT (0,1) |
| 10 | 1 | {1} | {-1} | {1} | c0 `1+0=1` ∈ anti, c1 cột, c2 `1-2=-1` ∈ diag, c3 | ĐẶT (1,3) |
| 11 | 2 | {1,3} | {-2,-1} | {1,4} | c0 | ĐẶT (2,0) |
| 12 | 3 | {0,1,3} | {-2,-1,2} | {1,2,4} | c0 cột, c1 cột, c2 | ĐẶT (3,2) |
| 13 | 4 | | | | | đủ 4 hậu → `[1, 3, 0, 2]` |

Để ý dòng 4, 7, 9: set đã được trả về **đúng** trạng thái cũ nhờ bước bỏ chọn.

In [16]:
def trace_n_queens_sets(n: int, stop_after: int = 1) -> None:
    queens, found = [], []
    cols, diag, anti = set(), set(), set()
    fmt = lambda s: '{' + ','.join(map(str, sorted(s))) + '}'

    def backtrack(r: int) -> None:
        if r == n:
            found.append(queens[:])
            print(f"r={r}: đủ {n} hậu => LỜI GIẢI {queens}")
            return
        head = f"r={r} cols={fmt(cols)} diag={fmt(diag)} anti={fmt(anti)}"
        notes = []
        for c in range(n):
            if c in cols:
                notes.append(f"c{c}:cột"); continue
            if r - c in diag:
                notes.append(f"c{c}:diag({r - c})"); continue
            if r + c in anti:
                notes.append(f"c{c}:anti({r + c})"); continue
            notes.append(f"c{c}:ĐẶT")
            print(f"{head:<44}| {' '.join(notes)}")
            notes = []
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            backtrack(r + 1)
            if len(found) >= stop_after:
                return                                   # chỉ để trace gọn
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()
        tried = ' '.join(notes) if notes else "(không còn cột)"
        print(f"{head:<44}| {tried} -> hết, quay lui")

    backtrack(0)

trace_n_queens_sets(4)

r=0 cols={} diag={} anti={}                 | c0:ĐẶT
r=1 cols={0} diag={0} anti={0}              | c0:cột c1:diag(0) c2:ĐẶT
r=2 cols={0,2} diag={-1,0} anti={0,3}       | c0:cột c1:anti(3) c2:cột c3:diag(-1) -> hết, quay lui
r=1 cols={0} diag={0} anti={0}              | c3:ĐẶT
r=2 cols={0,3} diag={-2,0} anti={0,4}       | c0:cột c1:ĐẶT
r=3 cols={0,1,3} diag={-2,0,1} anti={0,3,4} | c0:cột c1:cột c2:diag(1) c3:cột -> hết, quay lui
r=2 cols={0,3} diag={-2,0} anti={0,4}       | c2:diag(0) c3:cột -> hết, quay lui
r=1 cols={0} diag={0} anti={0}              | (không còn cột) -> hết, quay lui
r=0 cols={} diag={} anti={}                 | c1:ĐẶT
r=1 cols={1} diag={-1} anti={1}             | c0:anti(1) c1:cột c2:diag(-1) c3:ĐẶT
r=2 cols={1,3} diag={-2,-1} anti={1,4}      | c0:ĐẶT
r=3 cols={0,1,3} diag={-2,-1,2} anti={1,2,4}| c0:cột c1:cột c2:ĐẶT
r=4: đủ 4 hậu => LỜI GIẢI [1, 3, 0, 2]


## 10. Code nộp LeetCode — cách chính (3 set)

Trên LeetCode, `List` đã được import sẵn. Trong notebook ta đã `from typing import List` ở đầu.

In [17]:
class Solution:
    def solveNQueens(self, n: int) -> List[List[str]]:
        result = []
        queens = []
        cols, diag, anti = set(), set(), set()

        def backtrack(r: int) -> None:
            if r == n:
                result.append(['.' * c + 'Q' + '.' * (n - c - 1) for c in queens])
                return
            for c in range(n):
                if c in cols or (r - c) in diag or (r + c) in anti:
                    continue
                cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
                backtrack(r + 1)
                cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()

        backtrack(0)
        return result

### Bộ test

- Ví dụ đề: `n = 4`, `n = 1`. Đề cho phép thứ tự bất kỳ nên ta so sánh sau khi `sorted`.
- Biên: `n = 2`, `n = 3` không có lời giải → `[]`.
- Số lời giải đã biết cho `n = 1..9`: `1, 0, 0, 2, 10, 4, 40, 92, 352`.
- Mỗi bàn cờ trả về phải **hợp lệ**: kiểm bằng một hàm độc lập `is_valid_board` (đếm hậu theo hàng, cột, 2 chéo).

In [18]:
def is_valid_board(board: List[str]) -> bool:
    n = len(board)
    if any(len(row) != n or set(row) - {'Q', '.'} for row in board):
        return False
    pos = [(r, c) for r in range(n) for c in range(n) if board[r][c] == 'Q']
    if len(pos) != n:
        return False
    return (len({r for r, c in pos}) == n and len({c for r, c in pos}) == n
            and len({r - c for r, c in pos}) == n and len({r + c for r, c in pos}) == n)

sol = Solution()
assert sorted(sol.solveNQueens(4)) == sorted(example_1)
assert sol.solveNQueens(1) == [["Q"]]
assert sol.solveNQueens(2) == []
assert sol.solveNQueens(3) == []

known = [1, 0, 0, 2, 10, 4, 40, 92, 352]
for n in range(1, 10):
    boards = sol.solveNQueens(n)
    assert len(boards) == known[n - 1], n
    assert all(is_valid_board(b) for b in boards), n
    assert len(set(map(tuple, boards))) == len(boards), n      # không trùng
print("Tất cả test của Solution đều PASS!")

Tất cả test của Solution đều PASS!


### Đối chiếu chéo các cách (input ngẫu nhiên có `random.seed`)

Hai kiểm tra:

1. Chọn ngẫu nhiên các `n` trong `1..8`, so sánh kết quả (đã sort) của brute force, cách 2, cách 3.
2. Sinh ngẫu nhiên các list `queens` bất kỳ (có thể trùng cột), so sánh "hợp lệ theo quét" với "hợp lệ theo 3 set" — để chắc hai cách kiểm tra an toàn tương đương.

In [19]:
import random
random.seed(51)

for n in random.sample(range(1, 9), 6):        # 6 giá trị n khác nhau, chọn ngẫu nhiên
    a = sorted(n_queens_bruteforce(n))
    b = sorted(n_queens_backtrack_scan(n))
    c = sorted(n_queens_sets(n))
    assert a == b == c, n
    print(f"n = {n}: 3 cách cùng ra {len(a)} lời giải")

def valid_by_scan(q):
    return all(is_safe_scan(q[:r], r, q[r]) for r in range(len(q)))

def valid_by_sets(q):
    n = len(q)
    return (len(set(q)) == n and len({r - c for r, c in enumerate(q)}) == n
            and len({r + c for r, c in enumerate(q)}) == n)

for _ in range(2000):
    n = random.randint(1, 7)
    q = [random.randrange(n) for _ in range(n)]
    assert valid_by_scan(q) == valid_by_sets(q), q
print("2000 cấu hình ngẫu nhiên: hai cách kiểm tra an toàn luôn khớp")

n = 4: 3 cách cùng ra 2 lời giải
n = 7: 3 cách cùng ra 40 lời giải
n = 5: 3 cách cùng ra 10 lời giải
n = 6: 3 cách cùng ra 4 lời giải
n = 2: 3 cách cùng ra 0 lời giải
n = 1: 3 cách cùng ra 1 lời giải
2000 cấu hình ngẫu nhiên: hai cách kiểm tra an toàn luôn khớp


## 11. Vì sao đúng? (invariant)

**Invariant** (bất biến — điều luôn đúng mỗi khi vào `backtrack(r)`):

> Các hậu ở hàng `0..r-1` (lưu trong `queens`) đôi một không ăn nhau, và `cols`, `diag`, `anti` chứa **đúng** các giá trị `c`, `r-c`, `r+c` của những hậu đó — không thừa, không thiếu.

- Lúc bắt đầu `r = 0`: chưa có hậu, 3 set rỗng → đúng.
- Ta chỉ đặt `(r, c)` khi nó không trùng cột / chéo nào trong set → sau khi đặt, hàng `0..r` vẫn không ăn nhau, và ta thêm đúng 3 giá trị của nó → invariant đúng cho `backtrack(r + 1)`.
- Sau khi gọi xong, bước bỏ chọn gỡ đúng 3 giá trị đó → trạng thái trở lại như cũ, vòng `for` thử cột kế tiếp trên nền đúng.

**Không trả nhầm:** khi `r == n`, invariant nói `n` hậu đôi một không ăn nhau → đó là lời giải thật.

**Không bỏ sót:** mọi lời giải có đúng 1 hậu ở mỗi hàng, nên nó là một đường đi `queens[0], queens[1], ...` trên cây. Ta chỉ cắt nhánh khi phần đầu đã có 2 hậu ăn nhau — phần đầu như thế không bao giờ mở rộng thành lời giải được.

**Không trùng:** hai đường đi khác nhau trên cây cho hai list `queens` khác nhau → hai bàn cờ khác nhau.

## 12. Độ phức tạp của cách chính

Gọi `n` là kích thước bàn cờ, `S` là số lời giải.

- **Time:** số nút trên cây bị cắt tỉa không có công thức đóng; giới hạn trên thường ghi là **`O(n!)`** nút, mỗi nút chạy vòng `for` `n` cột với kiểm tra `O(1)`. Cộng thêm `O(S · n²)` để dựng các bàn cờ output. Thực tế nhanh hơn `n!` rất nhiều nhờ cắt tỉa.
- **Space:** `O(n)` cho `queens`, 3 set (mỗi set ≤ n phần tử) và độ sâu đệ quy `n` — **chưa tính** output (`S · n²` ký tự).

Với `n ≤ 9` (theo constraints) thì cả 3 cách đều chạy được, nhưng cách 3 là cách chuẩn để trình bày phỏng vấn và mở rộng sang bài lớn hơn.

Cell dưới đếm **số lần thử một ô** `(r, c)` của từng cách để thấy hiệu quả cắt tỉa (brute force: đếm số hoán vị × n hàng).

In [20]:
def count_cell_checks(n: int) -> int:
    # đếm số lần cách 2 / cách 3 thử một ô (r, c) (hai cách thử y hệt nhau, chỉ khác giá kiểm tra)
    checks = 0
    cols, diag, anti = set(), set(), set()
    def backtrack(r):
        nonlocal checks
        if r == n:
            return
        for c in range(n):
            checks += 1
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c)
            backtrack(r + 1)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c)
    backtrack(0)
    return checks

print(" n | brute force: n! * n | backtracking: số ô thử")
for n in range(4, 10):
    print(f"{n:>2} | {factorial(n) * n:>19,} | {count_cell_checks(n):>22,}")

 n | brute force: n! * n | backtracking: số ô thử
 4 |                  96 |                     60
 5 |                 600 |                    220
 6 |               4,320 |                    894
 7 |              35,280 |                  3,584
 8 |             322,560 |                 15,720
 9 |           3,265,920 |                 72,378


Ghi chú `nonlocal checks`: biến `checks` nằm ở hàm ngoài; muốn **gán lại** nó (`checks += 1`) bên trong hàm lồng `backtrack` thì phải khai báo `nonlocal`. Với `result.append(...)` hay `cols.add(...)` thì không cần, vì ta chỉ **sửa nội dung** object chứ không gán lại tên biến.

## 13. Cách 4 (nâng cao, tùy chọn) — Bitmask

### Nền tảng: số nguyên như một dãy bit

Một số nguyên có thể xem như dãy công tắc bật/tắt. Ta dùng **bit thứ `c`** để nói "cột `c` đã bị chiếm".

| Phép | Ý nghĩa | Ví dụ (bit viết từ phải sang trái: bit 0 ở cuối) |
|---|---|---|
| `1 << c` | số chỉ bật bit `c` | `1 << 2 = 0b0100` |
| `a \| b` | hợp (OR) | `0b0001 \| 0b0100 = 0b0101` |
| `a & b` | giao (AND) | `0b0101 & 0b0110 = 0b0100` |
| `~a & full` | đảo bit trong phạm vi `n` bit | `full = (1 << n) - 1 = 0b1111` |
| `x & -x` | lấy bit 1 thấp nhất (lowbit) | `0b0110 & -0b0110 = 0b0010` |
| `bit.bit_length() - 1` | vị trí của bit duy nhất | `0b0100` → `2` |
| `x << 1`, `x >> 1` | dịch trái / phải 1 bit | `0b0100 << 1 = 0b1000` |

In [21]:
n = 4
full = (1 << n) - 1
b = lambda x: format(x, f'0{n}b')
print("full      =", b(full))
print("1 << 2    =", b(1 << 2))
print("0101|0110 =", b(0b0101 | 0b0110), " 0101&0110 =", b(0b0101 & 0b0110))
print("~0101&full=", b(~0b0101 & full))
x = 0b0110
print("lowbit(0110) =", b(x & -x), "-> cột", (x & -x).bit_length() - 1)

full      = 1111
1 << 2    = 0100
0101|0110 = 0111  0101&0110 = 0100
~0101&full= 1010
lowbit(0110) = 0010 -> cột 1


### Ý tưởng bitmask cho N-Queens

Thay 3 set bằng 3 số nguyên, nhưng **tính theo hàng hiện tại**:

- `cols`: bit `c` bật nếu cột `c` đã có hậu.
- `dmask`: bit `c` bật nếu ô `(r, c)` của **hàng hiện tại** bị một đường chéo `\` ăn.
- `amask`: bit `c` bật nếu ô `(r, c)` của hàng hiện tại bị một đường chéo `/` ăn.

Khi đặt hậu ở cột `c` (bit `bit = 1 << c`) rồi xuống hàng dưới:

- Đường chéo `\` của nó đi xuống sang **cột `c + 1`** → `dmask` mới là `(dmask | bit) << 1` (cắt bớt bit vượt quá `n` bằng `& full`).
- Đường chéo `/` đi xuống sang **cột `c - 1`** → `amask` mới là `(amask | bit) >> 1`.

Các cột còn trống ở hàng hiện tại: `avail = full & ~(cols | dmask | amask)`. Lấy lần lượt từng bit thấp nhất của `avail` để thử (tức thử cột tăng dần, giống các cách trước).

Đây là `r - c` / `r + c` "trá hình": thay vì lưu mã đường chéo, ta lưu luôn **bóng** của đường chéo rơi xuống hàng đang xét.

In [22]:
def n_queens_bitmask(n: int) -> List[List[str]]:
    full = (1 << n) - 1
    result, queens = [], []

    def backtrack(cols: int, dmask: int, amask: int) -> None:
        if cols == full:                              # đủ n cột -> đủ n hậu
            result.append(build_board(queens, n))
            return
        avail = full & ~(cols | dmask | amask)        # các cột an toàn ở hàng này
        while avail:
            bit = avail & -avail                      # cột an toàn nhỏ nhất
            avail ^= bit                              # gạch bit đó khỏi avail
            queens.append(bit.bit_length() - 1)
            backtrack(cols | bit, ((dmask | bit) << 1) & full, (amask | bit) >> 1)
            queens.pop()

    backtrack(0, 0, 0)
    return result

print(n_queens_bitmask(4))
for n in range(1, 10):
    assert n_queens_bitmask(n) == n_queens_sets(n)     # cùng cả thứ tự
print("Bitmask khớp cách 3 (cả thứ tự) cho n = 1..9")

[['.Q..', '...Q', 'Q...', '..Q.'], ['..Q.', 'Q...', '...Q', '.Q..']]


Bitmask khớp cách 3 (cả thứ tự) cho n = 1..9


Bitmask có cùng độ phức tạp tiệm cận với cách 3, nhưng hằng số nhỏ hơn (phép bit rất rẻ, không cần vòng `for` qua cột bị chiếm). Nó hữu ích khi chỉ cần **đếm** (bài #52) với `n` lớn hơn. Với bài này, cách 3 dễ đọc và dễ giải thích hơn.

## 14. So sánh các cách

| Cách | Ý tưởng | Kiểm tra 1 ô | Time (ước lượng) | Space (không tính output) | Nên dùng khi |
|---|---|---|---|---|---|
| 1. Brute force hoán vị | thử mọi hoán vị, lọc chéo | (lọc cả hoán vị `O(n²)`) | `O(n! · n²)` | `O(n)` | hiểu đề, làm bộ kiểm chứng |
| 2. Backtracking + quét | đặt từng hàng, quét hậu đã đặt | `O(n)` | `O(n! · n)` | `O(n)` | học backtracking lần đầu |
| 3. Backtracking + 3 set | `cols`, `r-c`, `r+c` | `O(1)` | `O(n!)` | `O(n)` | **cách chính** để nộp/phỏng vấn |
| 4. Bitmask | 3 số nguyên, dịch bit | `O(1)`, chỉ duyệt ô trống | `O(n!)`, hằng số nhỏ | `O(n)` | đếm nhanh, muốn tối ưu |

(Time của cách 2–4 là giới hạn trên lỏng; thực tế nhỏ hơn rất nhiều nhờ cắt tỉa. Tất cả chưa tính `O(S · n²)` dựng output.)

## 15. Lỗi thường gặp

### Lỗi 1: Lưu `queens` (hoặc `path`) mà không copy

```python
if r == n:
    result.append(queens)      # SAI: lưu cùng một object
```

`queens` là **một list duy nhất** được sửa suốt quá trình. Mọi phần tử trong `result` đều trỏ tới cùng object đó; khi quay lui xong, nó bị `pop` về rỗng → kết quả là toàn list rỗng.

Sửa: lưu bản sao `queens[:]`, hoặc dựng luôn bàn cờ (list chuỗi mới) như `build_board(queens, n)`.

In [23]:
def bug_no_copy(n):
    result, queens = [], []
    cols, diag, anti = set(), set(), set()
    def backtrack(r):
        if r == n:
            result.append(queens)           # SAI
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            backtrack(r + 1)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()
    backtrack(0)
    return result

print("sai :", bug_no_copy(4))
print("đúng:", [[row.index('Q') for row in b] for b in n_queens_sets(4)])

sai : [[], []]
đúng: [[1, 3, 0, 2], [2, 0, 3, 1]]


### Lỗi 2: Quên (hoặc gỡ thiếu) bước bỏ chọn

Nếu chỉ `queens.pop()` mà quên `cols.remove(c)` ..., các set còn "nhớ" hậu đã nhấc ra → những ô thật ra an toàn bị coi là bị ăn → **mất lời giải** (output dưới: mất sạch).

### Lỗi 3: `return` sớm trong vòng `for`

Viết `return` ngay sau `backtrack(r + 1)` (vì nghĩ "đi tiếp xong là xong") sẽ khiến mỗi hàng chỉ thử **một** cột an toàn đầu tiên → bỏ sót lời giải (với n = 4, 6, 8 nhánh đầu tiên đều bế tắc nên mất sạch, như output dưới). Bài này cần **tất cả** lời giải, nên vòng `for` phải chạy hết.

In [24]:
def bug_forget_unchoose(n):
    result, queens = [], []
    cols, diag, anti = set(), set(), set()
    def backtrack(r):
        if r == n:
            result.append(build_board(queens, n))
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            backtrack(r + 1)
            queens.pop()                    # SAI: quên gỡ cols/diag/anti
    backtrack(0)
    return result

def bug_early_return(n):
    result, queens = [], []
    cols, diag, anti = set(), set(), set()
    def backtrack(r):
        if r == n:
            result.append(build_board(queens, n))
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            backtrack(r + 1)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()
            return                          # SAI: không thử các cột còn lại
    backtrack(0)
    return result

for n in [4, 6, 8]:
    print(f"n={n}: đúng {len(n_queens_sets(n)):>2} | quên bỏ chọn {len(bug_forget_unchoose(n))}"
          f" | return sớm {len(bug_early_return(n))}")

n=4: đúng  2 | quên bỏ chọn 0 | return sớm 0
n=6: đúng  4 | quên bỏ chọn 0 | return sớm 0
n=8: đúng 92 | quên bỏ chọn 0 | return sớm 0


### Lỗi 4: Dựng bàn cờ 2D bằng `[['.'] * n] * n`

Nhiều người dựng bàn cờ dạng list 2D rồi sửa ô. Nhưng `[['.'] * n] * n` tạo `n` **tham chiếu tới cùng một hàng** → sửa một ô thì cả cột đó ở mọi hàng đổi theo.

Sửa: `[['.'] * n for _ in range(n)]` (mỗi hàng là list mới), và nhớ `''.join(row)` để đổi về chuỗi như đề yêu cầu.

### Lỗi 5: Dùng mã đường chéo làm index list mà quên offset

Nếu thay set bằng list boolean `diag_used = [False] * (2*n - 1)`, thì `r - c` có thể âm. Python không báo lỗi với index âm (nó đếm từ cuối list!) nên bug rất khó thấy. Sửa: dùng `diag_used[r - c + n - 1]`.

In [25]:
n = 3
wrong = [['.'] * n] * n
wrong[0][1] = 'Q'
print("sai :", [''.join(row) for row in wrong])

right = [['.'] * n for _ in range(n)]
right[0][1] = 'Q'
print("đúng:", [''.join(row) for row in right])

diag_used = [False] * (2 * n - 1)
r, c = 0, 2
print("r - c =", r - c, "-> index đúng sau offset:", r - c + n - 1)

sai : ['.Q.', '.Q.', '.Q.']
đúng: ['.Q.', '...', '...']
r - c = -2 -> index đúng sau offset: 0


## 16. Mẹo nhận dạng pattern

- Đề hỏi **"return all ..."** (mọi cấu hình / mọi tổ hợp / mọi cách sắp) và constraints **rất nhỏ** (ở đây `n <= 9`) → nghĩ ngay tới **backtracking**.
- Có thể chia lời giải thành **chuỗi quyết định** có thứ tự (hàng 0, hàng 1, ...) → mỗi tầng đệ quy là một quyết định.
- Ràng buộc kiểu "**mỗi hàng / cột / đường chéo tối đa một**" → dùng `set` (hoặc bitmask) đánh dấu đường đã dùng; đường chéo đặt tên bằng `r - c` và `r + c`.
- Khuôn 3 bước luôn phải đối xứng: **chọn → đi tiếp → bỏ chọn**.
- Lưu kết quả bằng **bản sao** hoặc object mới.

## 17. Biến thể & bài liên quan

| Bài | Liên hệ |
|---|---|
| **#52 N-Queens II** | cùng thuật toán, chỉ **đếm** số lời giải: bỏ `queens` và `build_board`, thay bằng `count += 1` (bitmask rất hợp) |
| #46 Permutations | đúng bộ khung backtracking của mục 5 |
| #37 Sudoku Solver | backtracking + 3 nhóm ràng buộc (hàng, cột, ô 3x3), mỗi nhóm một set |
| #1001 Grid Illumination | dùng lại mẹo mã đường chéo `r - c`, `r + c` (đếm đèn theo hàng/cột/chéo) |
| Tìm **một** lời giải | dừng ngay khi gặp lời giải đầu tiên (Bài tập 5) |
| Tận dụng đối xứng | bàn cờ lật trái–phải vẫn là lời giải → chỉ thử nửa cột ở hàng 0 rồi phản chiếu (Bài tập 6) |

## 18. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** và **Đáp án** ở cuối notebook.

### Bài 1 — Tính tay (không chạy code)

Trên bàn 5x5, cho các ô: `A = (0, 3)`, `B = (2, 1)`, `C = (4, 3)`, `D = (3, 0)`.

a) Tính `r - c` và `r + c` cho từng ô.
b) Cặp nào ăn nhau theo đường chéo? Theo cột?

### Bài 2 — Đoán output

Thuật toán cách 3 thử cột theo thứ tự tăng dần. Với `n = 5`, lời giải **đầu tiên** được thêm vào `result` có `queens` bằng bao nhiêu? Hãy tự "chạy tay" như bảng trace mục 9 (gợi ý: hàng 0 đặt ở cột 0 vẫn có lời giải).

### Bài 3 — Điền TODO: backtracking 3 set

Hoàn thiện hàm theo đúng khuôn chọn → đi tiếp → bỏ chọn.

In [26]:
def exercise_n_queens(n: int) -> List[List[str]]:
    result, queens = [], []
    cols, diag, anti = set(), set(), set()

    def backtrack(r: int) -> None:
        # TODO 1: nếu đã đặt đủ n hàng -> thêm bàn cờ vào result, return
        # TODO 2: duyệt c từ 0 đến n-1
        # TODO 3: bỏ qua nếu c, r - c, r + c đã bị chiếm
        # TODO 4: chọn (cập nhật 3 set + queens), gọi backtrack(r + 1), bỏ chọn
        pass

    backtrack(0)
    return result

# Tự làm xong rồi mới mở test:
# assert len(exercise_n_queens(4)) == 2
# assert exercise_n_queens(1) == [["Q"]]
# assert len(exercise_n_queens(8)) == 92

### Bài 4 — Biến thể: chỉ đếm số lời giải (liên hệ #52)

Viết `count_n_queens(n)` trả về **số** lời giải, không dựng bàn cờ. Gợi ý: không cần `queens`, dùng `nonlocal` hoặc cho `backtrack` trả về số lời giải trong nhánh.

In [27]:
def count_n_queens(n: int) -> int:
    # TODO: backtracking 3 set, nhưng chỉ đếm
    raise NotImplementedError

# assert [count_n_queens(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]

### Bài 5 — Biến thể: chỉ cần MỘT lời giải

Viết `first_n_queens(n)` trả về bàn cờ của lời giải đầu tiên (theo thứ tự thử cột tăng dần), hoặc `[]` nếu không có. Phải **dừng ngay** khi tìm được, không duyệt tiếp.

Gợi ý: cho `backtrack(r)` trả về `True` khi đã tìm thấy để các tầng trên dừng theo.

In [28]:
def first_n_queens(n: int) -> List[str]:
    # TODO: backtrack trả về True/False, dừng sớm khi tìm thấy
    pass

# assert first_n_queens(4) == [".Q..", "...Q", "Q...", "..Q."]
# assert first_n_queens(3) == []
# assert is_valid_board(first_n_queens(9))

### Bài 6 — Mini challenge: tận dụng đối xứng

Lật bàn cờ trái–phải (cột `c` → `n - 1 - c`) biến lời giải thành lời giải. Hãy viết `n_queens_mirror(n)`:

- Hàng 0 chỉ thử các cột `c < n // 2`; mỗi lời giải tìm được, thêm cả nó **và** bản lật.
- Nếu `n` lẻ, còn phải thử riêng cột giữa `c = n // 2` ở hàng 0 (lời giải loại này **không** cần thêm bản lật, vì bản lật cũng bắt đầu ở cột giữa nên sẽ được tìm thấy riêng).

Kết quả (sau khi sort) phải bằng `sorted(n_queens_sets(n))`.

In [29]:
def n_queens_mirror(n: int) -> List[List[str]]:
    # TODO: backtracking như cách 3, nhưng giới hạn cột ở hàng 0 và phản chiếu
    pass

# for n in range(1, 10):
#     assert sorted(n_queens_mirror(n)) == sorted(n_queens_sets(n))

## 19. Gợi ý theo mức

**Bài 1** — (1) Cùng `r - c` thì cùng chéo `\`, cùng `r + c` thì cùng chéo `/`. (2) Lập bảng 4 dòng 2 cột rồi dò giá trị trùng. (3) `A`: `-3, 3`; ba ô còn lại tự tính.

**Bài 2** — (1) Bắt đầu `(0,0)`, mỗi hàng chọn cột nhỏ nhất chưa bị ăn. (2) Thử ngay: hàng 1 cột nhỏ nhất an toàn là 2. (3) Đáp số có dạng `[0, 2, ?, ?, ?]`, và lần này không cần quay lui.

**Bài 3** — (1) Chép khuôn mục 9. (2) Điều kiện bỏ qua có 3 vế nối bằng `or`. (3) Bước bỏ chọn dùng `remove` cho đúng 3 giá trị đã `add` và `queens.pop()`.

**Bài 4** — (1) Mỗi lần `r == n` là một lời giải. (2) `def backtrack(r) -> int`: trả về `1` khi `r == n`, còn lại cộng dồn kết quả các nhánh con. (3) `total += backtrack(r + 1)` giữa bước chọn và bỏ chọn.

**Bài 5** — (1) Khi nhánh con báo đã tìm thấy thì tầng hiện tại không cần thử cột khác. (2) `if backtrack(r + 1): return True`. (3) Khi tìm thấy, **không** cần bỏ chọn nữa vì `queens` đang giữ lời giải; dựng bàn cờ từ `queens` sau khi `backtrack(0)` trả về `True`.

**Bài 6** — (1) Viết `backtrack(r, limit_cols)` hoặc tham số `first_cols` cho hàng 0. (2) Hàm lật một bàn cờ: `[row[::-1] for row in board]`. (3) Chạy backtracking 2 lần: lần 1 hàng 0 dùng `range(n // 2)` và thêm cả bản lật; lần 2 (nếu `n` lẻ) hàng 0 chỉ dùng `[n // 2]`. Cẩn thận `n = 1`: `n // 2 = 0` nên lần 1 không chạy gì, lần 2 thử cột 0.

## 20. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1**

| Ô | `r - c` | `r + c` |
|---|---:|---:|
| A (0,3) | -3 | 3 |
| B (2,1) | 1 | 3 |
| C (4,3) | 1 | 7 |
| D (3,0) | 3 | 3 |

- Cùng `r + c = 3`: **A, B, D** đôi một ăn nhau theo chéo `/`.
- Cùng `r - c = 1`: **B và C** ăn nhau theo chéo `\`.
- Cùng cột 3: **A và C**.

**Bài 2:** `queens = [0, 2, 4, 1, 3]`, tức bàn cờ `["Q....", "..Q..", "....Q", ".Q...", "...Q."]` (đi thẳng một mạch, không quay lui lần nào). Cell dưới kiểm chứng Bài 1, Bài 2.

In [30]:
cells_ex = {'A': (0, 3), 'B': (2, 1), 'C': (4, 3), 'D': (3, 0)}
for name, (r, c) in cells_ex.items():
    print(name, "r-c =", r - c, " r+c =", r + c)

names = list(cells_ex)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        (r1, c1), (r2, c2) = cells_ex[names[i]], cells_ex[names[j]]
        if c1 == c2:
            print(names[i], names[j], "cùng cột")
        elif abs(r1 - r2) == abs(c1 - c2):
            print(names[i], names[j], "cùng chéo")

assert n_queens_sets(5)[0] == ["Q....", "..Q..", "....Q", ".Q...", "...Q."]
print("Bài 2: lời giải đầu tiên n=5 là", n_queens_sets(5)[0])

A r-c = -3  r+c = 3
B r-c = 1  r+c = 3
C r-c = 1  r+c = 7
D r-c = 3  r+c = 3
A B cùng chéo
A C cùng cột
A D cùng chéo
B C cùng chéo
B D cùng chéo
Bài 2: lời giải đầu tiên n=5 là ['Q....', '..Q..', '....Q', '.Q...', '...Q.']


In [31]:
# Đáp án Bài 3
def answer_exercise_n_queens(n: int) -> List[List[str]]:
    result, queens = [], []
    cols, diag, anti = set(), set(), set()

    def backtrack(r: int) -> None:
        if r == n:
            result.append(build_board(queens, n))
            return
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            backtrack(r + 1)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()

    backtrack(0)
    return result

assert len(answer_exercise_n_queens(4)) == 2
assert answer_exercise_n_queens(1) == [["Q"]]
assert len(answer_exercise_n_queens(8)) == 92


# Đáp án Bài 4
def answer_count_n_queens(n: int) -> int:
    cols, diag, anti = set(), set(), set()

    def backtrack(r: int) -> int:
        if r == n:
            return 1
        total = 0
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c)
            total += backtrack(r + 1)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c)
        return total

    return backtrack(0)

assert [answer_count_n_queens(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


In [32]:
# Đáp án Bài 5
def answer_first_n_queens(n: int) -> List[str]:
    queens = []
    cols, diag, anti = set(), set(), set()

    def backtrack(r: int) -> bool:
        if r == n:
            return True
        for c in range(n):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            if backtrack(r + 1):
                return True                    # giữ nguyên queens = lời giải
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()
        return False

    return build_board(queens, n) if backtrack(0) else []

assert answer_first_n_queens(4) == [".Q..", "...Q", "Q...", "..Q."]
assert answer_first_n_queens(3) == []
assert is_valid_board(answer_first_n_queens(9))


# Đáp án Bài 6
def answer_n_queens_mirror(n: int) -> List[List[str]]:
    result, queens = [], []
    cols, diag, anti = set(), set(), set()

    def backtrack(r: int, first_cols, add_mirror: bool) -> None:
        if r == n:
            board = build_board(queens, n)
            result.append(board)
            if add_mirror:
                result.append([row[::-1] for row in board])
            return
        for c in (first_cols if r == 0 else range(n)):
            if c in cols or (r - c) in diag or (r + c) in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)
            backtrack(r + 1, first_cols, add_mirror)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()

    backtrack(0, range(n // 2), True)            # nửa trái + bản lật
    if n % 2 == 1:
        backtrack(0, [n // 2], False)            # cột giữa, tự đối xứng nhóm
    return result

for n in range(1, 10):
    assert sorted(answer_n_queens_mirror(n)) == sorted(n_queens_sets(n)), n
print("Bài 5, Bài 6: PASS")

Bài 5, Bài 6: PASS


## 21. Template nên nhớ

```python
class Solution:
    def solveNQueens(self, n: int) -> List[List[str]]:
        result, queens = [], []
        cols, diag, anti = set(), set(), set()      # c, r - c, r + c

        def backtrack(r):
            if r == n:                              # đủ n hàng
                result.append(['.' * c + 'Q' + '.' * (n - c - 1) for c in queens])
                return
            for c in range(n):
                if c in cols or r - c in diag or r + c in anti:
                    continue
                cols.add(c); diag.add(r - c); anti.add(r + c); queens.append(c)        # chọn
                backtrack(r + 1)                                                       # đi tiếp
                cols.remove(c); diag.remove(r - c); anti.remove(r + c); queens.pop()   # bỏ chọn

        backtrack(0)
        return result
```

Câu thần chú: **"Mỗi hàng một hậu; cột, `r - c`, `r + c` vào set; chọn → đi tiếp → bỏ chọn."**

## 22. Checklist trước khi submit

- [ ] Đặt hậu **theo hàng** (không cần kiểm tra hàng), điểm dừng là `r == n`.
- [ ] Kiểm tra đủ 3 điều kiện: cột, `r - c`, `r + c`.
- [ ] Bước bỏ chọn gỡ **đúng và đủ** những gì bước chọn đã thêm.
- [ ] Không `return` sớm trong vòng `for` (đề cần **tất cả** lời giải).
- [ ] Kết quả lưu là object **mới** (list chuỗi mới hoặc `queens[:]`), không phải tham chiếu `queens`.
- [ ] Mỗi bàn cờ là `List[str]`, mỗi chuỗi dài đúng `n`, dùng `'Q'` và `'.'`.
- [ ] Thử tay `n = 1` → `[["Q"]]`, `n = 2, 3` → `[]`, `n = 4` → 2 lời giải.